In [13]:
# Ячейка 1
%matplotlib inline
import re
import os
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set(style="whitegrid")

# Параметры — проверь путь к файлу
INPUT_XLSX = Path("Бровки_now.xlsx")  # <-- поправь, если файл под другим именем
assert INPUT_XLSX.exists(), f"Файл не найден: {INPUT_XLSX}"

OUTPUT_DIR = Path("results_models")
(OUTPUT_DIR / "profiles").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# Формат файла (твоя жесткая структура)
HEADER_ROW_IDX = 1    # индекс строки с названиями столбцов (вторая строка Excel)
DATA_START_ROW = 2    # индекс строки, где начинаются данные (третья строка Excel)
COLS_PER_PROFILE = 5  # число колонок в одном профиле: [Дата измерения, ГП, ИГП, РГП-ПН, ИПН]
MAX_EMPTY_RUN = 3     # сколько подряд пустых строк означает конец блока

print("Input:", INPUT_XLSX)
print("Outputs ->", OUTPUT_DIR.resolve())


Input: Бровки_now.xlsx
Outputs -> C:\Users\user\Desktop\Diplom\po\results_models


In [14]:
# Ячейка 2
def safe_str(x):
    return "" if pd.isna(x) else str(x).strip()

def clean_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(r'\s+', '', regex=True)
         .str.replace('*', '', regex=False)
         .str.replace(',', '.', regex=False)
         .str.replace('−', '-', regex=False)
         .str.replace(r'[^0-9\.\-]', '', regex=True),
        errors='coerce'
    )

def make_unique(names):
    cnt = Counter()
    out = []
    for n in names:
        key = "" if n is None else str(n)
        if cnt[key] == 0:
            out.append(key)
        else:
            out.append(f"{key}__{cnt[key]}")
        cnt[key] += 1
    return out

def safe_filename(s):
    return re.sub(r'[^\w\d\-]', '_', str(s))[:120]


In [15]:
# Ячейка 3
def parse_sheet_fixed(sheet_df,
                      header_row_idx=HEADER_ROW_IDX,
                      data_start_row=DATA_START_ROW,
                      cols_per_profile=COLS_PER_PROFILE,
                      max_empty_run=MAX_EMPTY_RUN):
    """
    Возвращает profiles_dict и combined_df для листа.
    profiles_dict: {profile_title: df_profile}
    combined_df: объединённый DF со столбцом 'Профиль'
    """
    ncols = sheet_df.shape[1]
    n_profiles = max(0, ncols // cols_per_profile)
    print("profiles")
    profiles = {}
    combined_parts = []
    title_row = sheet_df.iloc[0] if sheet_df.shape[0] > 0 else None
    expected_headers = ['Дата измерения','ГП','ИГП, м','РГП-ПН, м','ИПН, м']

    for i in range(n_profiles):
        start = i * cols_per_profile
        end = min(start + cols_per_profile, ncols)
        block = sheet_df.iloc[:, start:end].copy()

        # возьмём заголовки из строки header_row_idx
        raw_colnames = block.iloc[header_row_idx].astype(str).fillna("").tolist()
        if all(safe_str(x) == "" for x in raw_colnames):
            raw_colnames = expected_headers[:block.shape[1]]

        # нормализуем имена колонок на короткие
        norm_names = []
        print("OK")
        for j, nm in enumerate(raw_colnames):
            s = safe_str(nm).lower()
            if 'дата' in s:
                norm_names.append('Дата')
            elif s.strip() == 'гп' or s.strip() == 'gp':
                norm_names.append('ГП')
            elif 'игп' in s or 'измер' in s:
                norm_names.append('ИГП_м')
            elif 'ргп' in s or ('пн' in s and 'ргп' in s):
                norm_names.append('РГП_ПН_м')
            elif 'ипн' in s or 'бров' in s:
                norm_names.append('ИПН_м')
                print("ОК")
            else:
                # fallback by position
                mapping = {0:'Дата',1:'ГП',2:'ИГП_м',3:'РГП_ПН_м',4:'ИПН_м'}
                norm_names.append(mapping.get(j, f"col_{j}"))
        norm_names = make_unique(norm_names)
        block.columns = norm_names

        # данные — начиная со строки data_start_row
        rows = []
        empty_run = 0
        for r in range(data_start_row, sheet_df.shape[0]):
            row = block.iloc[r].values.tolist()
            if all(safe_str(x) == "" for x in row):
                empty_run += 1
                if empty_run >= max_empty_run:
                    break
                else:
                    continue
            empty_run = 0
            rows.append(row)
        if not rows:
            df = pd.DataFrame(columns=block.columns)
        else:
            df = pd.DataFrame(rows, columns=block.columns)

        # приведение типов
        # Дата
        date_col = next((c for c in df.columns if 'дата' in c.lower()), None)
        if date_col is None:
            for c in df.columns[:min(3, len(df.columns))]:
                if pd.to_datetime(df[c], errors='coerce', dayfirst=True).notna().sum() > 0:
                    date_col = c; break
        if date_col:
            df['Дата'] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
        else:
            df['Дата'] = pd.NaT

        # числа
        if 'ИГП_м' in df.columns:
            df['ИГП_м'] = clean_num_series(df['ИГП_м'])
        else:
            for c in df.columns:
                if 'игп' in c.lower() or 'измер' in c.lower():
                    df['ИГП_м'] = clean_num_series(df[c]); break

        if 'РГП_ПН_м' in df.columns:
            df['РГП_ПН_м'] = clean_num_series(df['РГП_ПН_м'])
        else:
            for c in df.columns:
                if 'ргп' in c.lower() or ('пн' in c.lower() and 'ргп' not in c.lower()):
                    df['РГП_ПН_м'] = clean_num_series(df[c]); break

        if 'ИПН_м' in df.columns:
            df['ИПН_м'] = clean_num_series(df['ИПН_м'])
        else:
            for c in df.columns:
                if 'ипн' in c.lower() or 'бров' in c.lower():
                    df['ИПН_м'] = clean_num_series(df[c]); break

        if 'ГП' in df.columns:
            df['ГП'] = df['ГП'].astype(str).str.strip().replace("nan","")

        # удалить строки без даты
        if 'Дата' in df.columns:
            df = df.dropna(subset=['Дата']).reset_index(drop=True)
        else:
            df = df.iloc[0:0]

        # год
        if 'Дата' in df.columns:
            df['Год'] = df['Дата'].dt.year
        else:
            df['Год'] = np.nan

        # название профиля
        profile_title = None
        try:
            if title_row is not None:
                v = safe_str(title_row.iloc[start])
                if v:
                    profile_title = v
        except Exception:
            profile_title = None
        if not profile_title:
            profile_title = f"Профиль_{i+1}"

        df_profile = df[[c for c in ['Дата','Пункт','ГП','ИГП_м','РГП_ПН_м','ИПН_м','Год'] if c in df.columns]].copy()
        df_profile.attrs['start_col'] = start
        df_profile.attrs['profile_title'] = profile_title

        profiles[profile_title] = df_profile
        cp = df_profile.copy()
        cp['Профиль'] = profile_title
        combined_parts.append(cp)

    combined = pd.concat(combined_parts, ignore_index=True) if combined_parts else pd.DataFrame()
    return profiles, combined


In [5]:
# Ячейка 4
xls = pd.ExcelFile(INPUT_XLSX)
sites_dict = {}
for sheet in xls.sheet_names:
    raw = pd.read_excel(INPUT_XLSX, sheet_name=sheet, header=None)
    profiles, combined = parse_sheet_fixed(raw)
    if not combined.empty:
        combined['Участок'] = sheet
    sites_dict[sheet] = {'profiles': profiles, 'combined': combined}

# объединённая таблица всех наблюдений
df_observations = pd.concat([v['combined'] for v in sites_dict.values() if not v['combined'].empty], ignore_index=True) if any(v['combined'].shape[0]>0 for v in sites_dict.values()) else pd.DataFrame()
df_observations.to_excel(OUTPUT_DIR / "df_observations.xlsx", index=False)
print("Parsed sheets:", len(sites_dict))
print("Total observations:", len(df_observations))

# экспорт каждого профиля в отдельный xlsx
for site, info in sites_dict.items():
    for prof_name, dfp in info['profiles'].items():
        if dfp.empty: 
            continue
        out_name = OUTPUT_DIR / "profiles" / f"{safe_filename(site)}__{safe_filename(prof_name)}.xlsx"
        dfp.to_excel(out_name, index=False)
print("Profiles exported to:", (OUTPUT_DIR / "profiles").resolve())


Parsed sheets: 10
Total observations: 680
Profiles exported to: C:\Users\user\Desktop\Diplom\po\results_models\profiles


In [6]:
# Ячейка 5
def compute_slope_retreat(years, values):
    years = np.array(years, dtype=float)
    values = np.array(values, dtype=float)
    mask = ~np.isnan(years) & ~np.isnan(values)
    if mask.sum() < 2:
        return np.nan
    slope = np.polyfit(years[mask], values[mask], 1)[0]
    return -float(slope)  # положительная -> отступание м/год

profiles_rows = []
for site, info in sites_dict.items():
    for prof_name, dfp in info['profiles'].items():
        if dfp is None or dfp.empty:
            continue
        ycol = 'ИПН_м' if ('ИПН_м' in dfp.columns and dfp['ИПН_м'].notna().sum()>=2) else None
        retreat = compute_slope_retreat(dfp['Год'], dfp[ycol]) if ycol else np.nan
        rows = {
            'Участок': site,
            'Профиль': prof_name,
            'n_obs': int(len(dfp)),
            'year_first': int(dfp['Год'].min()) if dfp['Год'].notna().any() else np.nan,
            'year_last': int(dfp['Год'].max()) if dfp['Год'].notna().any() else np.nan,
            'duration_years': int(dfp['Год'].max() - dfp['Год'].min()) if dfp['Год'].notna().any() else np.nan,
            'ycol_used': ycol,
            'mean_IPN': float(dfp[ycol].mean()) if ycol and dfp[ycol].notna().any() else np.nan,
            'std_IPN': float(dfp[ycol].std()) if ycol and dfp[ycol].notna().any() else np.nan,
            'retreat_m_per_year': float(retreat) if not np.isnan(retreat) else np.nan
        }
        profiles_rows.append(rows)

df_profiles_ipn = pd.DataFrame(profiles_rows)
df_profiles_ipn.to_excel(OUTPUT_DIR / "df_profiles_ipn.xlsx", index=False)
df_profiles_ipn.to_csv(OUTPUT_DIR / "df_profiles_ipn.csv", index=False)
print("Profile summary saved:", OUTPUT_DIR / "df_profiles_ipn.xlsx")
df_profiles_ipn.head()


Profile summary saved: results_models\df_profiles_ipn.xlsx


,Участок,Профиль,n_obs,year_first,year_last,duration_years,ycol_used,mean_IPN,std_IPN,retreat_m_per_year
0,Бережновка,ПРОФИЛЬ № 60,24,1958,2023,65,ИПН_м,108.174583,63.325368,3.176220
1,Бережновка,ПРОФИЛЬ № 61,24,1958,2023,65,ИПН_м,91.362083,77.396049,3.870345
2,Бережновка,ПРОФИЛЬ № 62,24,1958,2023,65,ИПН_м,226.794792,99.586717,4.849923
3,Бурты,ПРОФИЛЬ № 1,25,1987,2023,36,ИПН_м,61.444400,9.176637,0.984297
4,Бурты,ПРОФИЛЬ № 2,25,1987,2023,36,ИПН_м,51.828400,11.180792,1.192329


In [7]:
# Ячейка 6
# По участку: среднее ИПН по годам -> лин. тренд + 20 лет вперед
n_site_plots = 0
if df_observations.empty or 'ИПН_м' not in df_observations.columns:
    print("Нет данных ИПН в df_observations для построения site-level прогнозов.")
else:
    site_year = df_observations.groupby(['Участок','Год'])['ИПН_м'].mean().reset_index()
    for site, g in site_year.groupby('Участок'):
        g = g.dropna(subset=['Год','ИПН_м']).sort_values('Год')
        if g.shape[0] < 2:
            continue
        x = g['Год'].values; y = g['ИПН_м'].values
        coef = np.polyfit(x,y,1); trend = np.poly1d(coef)
        last_year = int(x.max()); future_years = np.arange(last_year+1, last_year+21)
        plt.figure(figsize=(8,4))
        plt.plot(x,y,'o-', label='Среднее ИПН (факт)')
        plt.plot(np.concatenate([x, future_years]), np.concatenate([trend(x), trend(future_years)]), 'r--', label=f'Прогноз 20 лет (slope={coef[0]:.4f} m/yr)')
        plt.title(f"Участок: {site}")
        plt.xlabel("Год"); plt.ylabel("ИПН, м"); plt.grid(True); plt.legend()
        fn = OUTPUT_DIR / "plots" / f"site_forecast_{safe_filename(site)}.png"
        plt.tight_layout(); plt.savefig(fn, dpi=150); plt.close()
        n_site_plots += 1
print("Site-level forecast plots saved:", n_site_plots)

# По профилю: ИПН и прогноз 20 лет (каждый профиль)
n_prof_plots = 0
for site, info in sites_dict.items():
    for prof_name, dfp in info['profiles'].items():
        if dfp is None or dfp.empty or 'ИПН_м' not in dfp.columns or dfp['ИПН_м'].notna().sum() < 2:
            continue
        d = dfp.dropna(subset=['Год','ИПН_м']).sort_values('Год')
        x = d['Год'].values; y = d['ИПН_м'].values
        coef = np.polyfit(x,y,1); trend = np.poly1d(coef)
        last_year = int(x.max()); future_years = np.arange(last_year+1, last_year+21)
        plt.figure(figsize=(7,4))
        plt.plot(x,y,'o-', label='Факт (ИПН)')
        plt.plot(np.concatenate([x, future_years]), np.concatenate([trend(x), trend(future_years)]), 'r--', label='Прогноз 20 лет')
        plt.title(f"{site} / {prof_name}")
        plt.xlabel("Год"); plt.ylabel("ИПН, м"); plt.grid(True); plt.legend()
        fn = OUTPUT_DIR / "plots" / f"profile_forecast_{safe_filename(site)}__{safe_filename(prof_name)}.png"
        plt.tight_layout(); plt.savefig(fn, dpi=150); plt.close()
        n_prof_plots += 1
print("Per-profile forecast plots saved:", n_prof_plots)


Site-level forecast plots saved: 10
Per-profile forecast plots saved: 33


In [8]:
# Ячейка 7
# Подготовка ML-матрицы (используем df_profiles_ipn)
if df_profiles_ipn.empty:
    print("df_profiles_ipn пуст — пропускаем ML.")
else:
    ml_df = df_profiles_ipn.dropna(subset=['retreat_m_per_year']).reset_index(drop=True)
    print("Profiles with target (IPN):", len(ml_df))
    feat_cols = ['n_obs','duration_years','year_first','year_last','mean_IPN','std_IPN']
    feat_cols = [c for c in feat_cols if c in ml_df.columns]
    X = ml_df[feat_cols].fillna(0).astype(float)
    y = ml_df['retreat_m_per_year'].astype(float)

    if len(ml_df) < 3:
        # демо обучение MLP
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
        pipe = Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(32,16), max_iter=1500, random_state=42))])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        metrics_demo = {
            'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred))),
            'MAE': float(mean_absolute_error(y_test, y_pred)),
            'R2': float(r2_score(y_test, y_pred))
        }
        joblib.dump(pipe, OUTPUT_DIR / "models" / "mlp_profiles_ipn_demo.joblib")
        pd.DataFrame([metrics_demo]).to_excel(OUTPUT_DIR / "models" / "mlp_profiles_ipn_demo_metrics.xlsx", index=False)
        print("Demo MLP trained and saved. Metrics:", metrics_demo)
    else:
        # train/test
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
        pipes = {
            'MLP': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(64,32), max_iter=1500, random_state=42))]),
            'RF': Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor(n_estimators=200, random_state=42))]),
            'GB': Pipeline([('scaler', StandardScaler()), ('gb', GradientBoostingRegressor(n_estimators=200, random_state=42))])
        }
        metrics_report = {}
        trained = {}
        for name, pipe in pipes.items():
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)
            m = {
                'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred))),
                'MAE': float(mean_absolute_error(y_test, y_pred)),
                'R2': float(r2_score(y_test, y_pred))
            }
            metrics_report[name] = m
            trained[name] = pipe
            joblib.dump(pipe, OUTPUT_DIR / "models" / f"{name}_profiles_ipn.joblib")
            print(f"Trained {name}, R2={m['R2']:.3f}, RMSE={m['RMSE']:.3f}")
        pd.DataFrame.from_dict(metrics_report, orient='index').to_excel(OUTPUT_DIR / "models" / "ml_models_profiles_ipn_metrics.xlsx")
        print("Models and metrics saved.")

        # GridSearch для RF (если образцов >=6)
        if len(ml_df) >= 6:
            print("Running GridSearchCV for RF...")
            rf = RandomForestRegressor(random_state=42)
            pipe_rf = Pipeline([('scaler', StandardScaler()), ('rf', rf)])
            param_grid = {
                'rf__n_estimators': [100, 200, 400],
                'rf__max_depth': [None, 5, 10],
                'rf__min_samples_split': [2, 4]
            }
            cv = min(5, max(2, len(ml_df)//2))
            grid = GridSearchCV(pipe_rf, param_grid, cv=cv, scoring='r2', n_jobs=-1)
            grid.fit(X, y)
            print("GridSearch best:", grid.best_params_, "best_score:", grid.best_score_)
            joblib.dump(grid.best_estimator_, OUTPUT_DIR / "models" / "rf_profiles_ipn_grid_best.joblib")
            # CV metrics
            r2s = cross_val_score(grid.best_estimator_, X, y, cv=cv, scoring='r2', n_jobs=-1)
            rmse_cv = np.sqrt(-cross_val_score(grid.best_estimator_, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1))
            metrics_grid = {'R2_mean': float(r2s.mean()), 'R2_std': float(r2s.std()), 'RMSE_mean': float(rmse_cv.mean()), 'RMSE_std': float(rmse_cv.std())}
            pd.DataFrame([metrics_grid]).to_excel(OUTPUT_DIR / "models" / "rf_grid_cv_metrics.xlsx", index=False)
            print("Grid CV metrics saved.")


Profiles with target (IPN): 33
Trained MLP, R2=0.964, RMSE=0.289
Trained RF, R2=0.963, RMSE=0.294
Trained GB, R2=0.954, RMSE=0.326
Models and metrics saved.
Running GridSearchCV for RF...
GridSearch best: {'rf__max_depth': 5, 'rf__min_samples_split': 2, 'rf__n_estimators': 100} best_score: -1.9262160183706514
Grid CV metrics saved.


In [9]:
# Ячейка 9
# Сохраним ключевые артефакты
if 'df_observations' in globals() and not df_observations.empty:
    df_observations.to_excel(OUTPUT_DIR / "df_observations_final.xlsx", index=False)
if 'df_profiles_ipn' in globals() and not df_profiles_ipn.empty:
    df_profiles_ipn.to_excel(OUTPUT_DIR / "df_profiles_ipn_final.xlsx", index=False)

# простой JSON-отчёт
report = {
    'n_sheets': len(sites_dict),
    'n_observations': int(len(df_observations)) if 'df_observations' in globals() else 0,
    'n_profiles': int(len(df_profiles_ipn)) if 'df_profiles_ipn' in globals() else 0
}
with open(OUTPUT_DIR / "run_summary.json", "w", encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Artifacts written to:", OUTPUT_DIR.resolve())
print(" - plots:", (OUTPUT_DIR / "plots").resolve())
print(" - profiles:", (OUTPUT_DIR / "profiles").resolve())
print(" - models:", (OUTPUT_DIR / "models").resolve())
print(" - summary:", OUTPUT_DIR / "run_summary.json")


Artifacts written to: C:\Users\user\Desktop\Diplom\po\results_models
 - plots: C:\Users\user\Desktop\Diplom\po\results_models\plots
 - profiles: C:\Users\user\Desktop\Diplom\po\results_models\profiles
 - models: C:\Users\user\Desktop\Diplom\po\results_models\models
 - summary: results_models\run_summary.json
